# OncoSeg —— 一次引导式、自包含的项目全景之旅

> 🌐 **English version**: `notebooks/Colab_Verify_Fixes.ipynb`（完全相同的 notebook，英文版）。

**初次接触？请从头读到尾。** 这个单一 notebook 讲清楚*OncoSeg 是什么、模型如何构建、以及每一步在做什么*——配有图示、公式、可运行代码和真实结果图像——然后在 Colab **GPU** 上端到端地验证代码。§1–§14 无需任何数据集或 checkpoint（模型图示已随仓库提交；真实训练运行是可选的最后一节）。

**运行前：** `Runtime → Change runtime type → Hardware accelerator → GPU`，然后 **Runtime → Run all**。

**路线图：**
1–2. 环境搭建（克隆 + 安装，一次性内核重启）。  
3–7. **理解它：** OncoSeg 是什么 · 数据 · 架构（+ 公式）· 构建并运行模型 · 训练损失。  
8. **查看真实结果：** 脑部 MRI 上的分割、准确率、不确定性。  
9–13. **验证它：** 完整测试套件、lint、冒烟测试、诚实的统计、以及一个实时 RECIST 疗效评估演示（附图）。  
14. 可选的真实训练。  15. 回顾 + 诚实的局限性。

> **一次性内核重启（预期之内）。** §1 会安装 `monai`/`numpy`；Colab 会在内存中保留旧的 numpy，这将导致后续导入失败。因此 §1 会**将内核重启一次**——当它重启时，只需**再次 Run all**（此操作是幂等的）。展示图示的小节在内核内运行；测试小节则作为子进程运行。


## 1 · 克隆 + 安装 + （一次性）内核重启

安装 `.[dev,serve,dicom]` —— `monai[all]`、`nibabel`、`fastapi`、`python-multipart`、`pydicom`/`highdicom`、
`pytest`、`ruff`（与 CI 使用的附加依赖相同，另加 `dicom`）。首次运行约需 2–3 分钟。

In [ ]:
import os
REPO = "https://github.com/danielchen26/OncoSeg-3D-Multi-Scale-Tumor-Segmentation-for-Automated-Treatment-Response-Assessment.git"
BRANCH = "fix/review-findings"
FLAG = "/content/.oncoseg_installed"   # 文件标志能在内核重启后保留（环境变量则不能）
if not os.path.isdir("/content/oncoseg"):
    !git clone --branch $BRANCH --depth 1 $REPO /content/oncoseg
%cd /content/oncoseg
!git log --oneline -1
if not os.path.exists(FLAG):
    !pip -q install -e "/content/oncoseg[dev,serve,dicom]"
    open(FLAG, "w").close()
    print("\n>>> 安装完成。正在重启内核一次，以便加载新版 numpy/monai —— 然后请再次运行 Runtime ▸ Run all。 <<<")
    import time; time.sleep(1)
    os.kill(os.getpid(), 9)   # 强制重启 Colab 内核
else:
    print("依赖已安装且内核已重启 —— 继续执行。")

## 2 · 运行时 + 导入检查（重启后）


In [ ]:
import os; os.chdir('/content/oncoseg') if os.path.isdir('/content/oncoseg') else None
import sys, platform
print('Python:', sys.version.split()[0], '|', platform.platform())
import torch
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('--- 可选依赖 ---')
for m in ['monai','nibabel','fastapi','pydicom','highdicom','scipy','numpy']:
    try: __import__(m); print(f'  {m}: OK')
    except Exception as e: print(f'  {m}: 缺失 ({e})')

## 3 · OncoSeg 是什么？（临床问题与流水线）


**OncoSeg** 能够自动测量医学影像中的肿瘤,以评估癌症治疗是否有效。当患者接受化疗或免疫治疗时,医生需要知道:*肿瘤是在缩小、保持稳定,还是在生长?* 如今,这一判断依靠**在 CT/MRI 扫描上手动测量肿瘤**——而这一过程:

- **耗时**:每位患者每次扫描需 15–30 分钟
- **主观**:两位放射科医生常常对肿瘤边界的位置意见不一(边缘定位差异达 20–40%)
- **范围受限**:传统方法仅测量单一的 2D 直径,忽略了完整的 3D 范围

OncoSeg 解决了这全部三个问题。它**自动分割 3D 肿瘤**,标记出**模型不确定的位置**(供放射科医生复核),并报告**标准的治疗反应类别**——全部在数秒内完成。

---

## 临床问题:手动 RECIST 测量

在肿瘤学试验中,肿瘤反应采用 **RECIST 1.1** 标准(实体瘤疗效评价标准,Response Evaluation Criteria In Solid Tumors)进行评估。工作流程如下:

1. 放射科医生加载基线 MRI(治疗开始时)
2. 手动勾画肿瘤边界
3. 测量轴向平面上的最长直径
4. 数周后对随访扫描重复此过程
5. 比较直径以对反应进行分类:
   - **CR**(完全缓解):肿瘤消失
   - **PR**(部分缓解):直径缩小 ≥30%
   - **SD**(疾病稳定):变化轻微(介于 PR 和 PD 阈值之间)
   - **PD**(疾病进展):直径增大 ≥20%(或较最低点增长 ≥5mm)

**问题在于:**这一过程劳动强度大、易出错,且未能利用现代深度学习。放射科医生必须正确分割肿瘤才能得到准确的测量结果,而阅片者间的差异意味着同一次扫描会因测量者不同而得出不同的结论。

---

## OncoSeg 的端到端流程

以下是 OncoSeg 自动完成的工作:

```
4-Channel MRI Input (T1, T1c, T2, FLAIR)
              │
              ▼
    ┌─────────────────────┐
    │  3D Swin Transformer│   ← 编码器:学习跨所有尺度的
    │   + CNN Decoder     │      空间模式
    └────────┬────────────┘
             │
             ▼
    ┌─────────────────────┐
    │   3D Segmentation   │   ← 以原始分辨率输出肿瘤
    │   Mask              │      (3 个嵌套区域)
    └────────┬────────────┘
             │
             ▼
    ┌─────────────────────┐
    │   Uncertainty Map   │   ← 逐体素置信度
    │  (MC Dropout)       │      标记出模糊区域
    └────────┬────────────┘
             │
             ▼
    ┌─────────────────────┐
    │  RECIST 1.1         │   ← 测量最长直径、
    │  Measurement        │      体积,按病灶逐个统计
    └────────┬────────────┘
             │
             ▼
    Response Category (CR/PR/SD/PD)
```

每一步骤将在下文说明。

---

## 输入:4 通道 3D MRI

模型接受**4 种配准好的 MRI 模态**,它们对脑肿瘤评估都不可或缺:

| Channel | Modality | What It Shows | Includes |
|---------|----------|---------------|----------:|
| 1 | **T1**(T1 加权) | 解剖学基线 | 健康组织、强化区域 |
| 2 | **T1c**(T1 + 钆造影剂) | 血脑屏障破坏 | 强化(灌注)肿瘤 |
| 3 | **T2**(T2 加权) | 自由水 / 水肿 | 肿瘤 + 周围肿胀 |
| 4 | **FLAIR**(液体衰减反转恢复) | 抑制脑脊液 | 水肿更清晰;将实体肿瘤与液体区分开 |

它们被堆叠成一个 `[B, 4, H, W, D]` 张量。模型学习融合这全部四者的信息:例如,如果某个区域在 T1c 和 T2 上明亮但在 FLAIR 上暗淡,那它很可能是*强化的实体肿瘤*;如果它在 FLAIR 和 T2 上明亮但在 T1c 上不亮,那它很可能是*水肿*。

---

## 输出:3 个嵌套的 BraTS 区域

肿瘤并非铁板一块——它包含对治疗反应不同、预后也不同的各个区域。OncoSeg 分割**三个嵌套类别**(遵循 BraTS 惯例):

```
┌────────────────────────────┐
│   Whole Tumor (WT) [ch=1]  │  所有肿瘤细胞
│                            │
│  ┌──────────────────────┐  │
│  │ Tumor Core (TC) [ch=0]
│  │                      │  │  坏死 + 强化
│  │  ┌────────────────┐  │  │
│  │  │ Enhancing (ET) │  │  │  仅灌注/存活部分
│  │  │  [ch=2]        │  │  │
│  │  └────────────────┘  │  │
│  │                      │  │
│  └──────────────────────┘  │
└────────────────────────────┘
```

三个**输出通道**为:
1. **TC**(肿瘤核心):非强化肿瘤 + 强化肿瘤(坏死或实性核心)
2. **WT**(全肿瘤):所有肿瘤(核心 + 周围水肿)
3. **ET**(强化肿瘤):仅指主动灌注、造影剂强化的区域(通常是最具侵袭性的部分)

这种多区域输出很重要,因为:
- **WT** 决定总体肿瘤负荷(用于 RECIST 的最大测量值)
- **TC** 指示肿瘤的实性范围(预后标志)
- **ET** 标记最具侵袭性的区域(指导放射治疗的靶向定位)

模型**同时且联合地**输出这些结果,而非按顺序逐个输出——它学习到有利于全部三项任务的共享表征。

---

## 输出:不确定性图(模型对自身存疑之处)

除了原始分割之外,OncoSeg 还通过多次运行模型(蒙特卡洛 Dropout,Monte Carlo Dropout)——每次施加略有不同的内部噪声——生成一张**逐体素不确定性图**。高不确定性标记出模糊的肿瘤边界——即以下这些区域:
- MRI 信号有噪声
- 肿瘤与正常组织之间的边界确实是渐变的(真实的生物学边缘)
- 多位放射科医生会各执一词

放射科医生可利用这张图来**优先安排人工复核**——将精力集中在高不确定性区域,而非整幅分割。

---

## 从分割到反应:RECIST 1.1 测量

一旦生成分割掩膜,OncoSeg 便自动提取 **RECIST 1.1 测量值**:

1. 通过连通分量标记**识别各个病灶**
2. **测量每个病灶的最长轴向直径**(跨所有切片、沿平面内方向的最大 Feret 距离)——不仅是最大切片上的,而是跨所有切片
3. 将病灶**筛选**为仅保留 ≥10 mm 的那些(RECIST 入选阈值)
4. **保留最多 5 个最大的病灶**(按 RECIST 1.1 全身上限)
5. **对它们的最长直径求和**(SLD = 最长直径之和,Sum of Longest Diameters)

在**基线(治疗开始时)**和**随访(后续时间点)**:
- 如果所有靶病灶消失 → **CR**(完全缓解)
- 如果 SLD 较基线减少 ≥30% → **PR**(部分缓解)
- 如果 SLD 较最低点增大 ≥20% 且绝对值增大 ≥5mm → **PD**(疾病进展)
- 否则 → **SD**(疾病稳定)

这一分类是**客观、可复现且自动化的**。

---

## 这在临床上为何重要

OncoSeg 解决了肿瘤学中一个真实的瓶颈:

| Aspect | Manual | OncoSeg |
|--------|--------|---------|
| **每次扫描耗时** | 15–30 分钟 | <1 秒 |
| **可复现性** | 受阅片者差异影响(20–40%) | 确定性(相同输入 → 相同输出) |
| **范围** | 单一 2D 直径 | 完整 3D 分割 + 多项指标 |
| **置信信号** | 无——临床医生必须信任测量结果 | 不确定性图标记出模糊区域 |
| **跨试验一致性** | 因机构、阅片者而异 | 全球标准化 |

在一项多中心试验中,200 名患者每 4 周扫描一次、持续 1 年(12 个时间点),手动测量 = **3600–7200 放射科医生工时**。OncoSeg 将其缩减至**不到 1 小时**(主要用于不确定性复核),从而解放放射科医生去处理其他任务并缩短试验周期。

---

## 你将在本 notebook 中看到什么

本 notebook 在来自医学分割十项全能赛(Medical Segmentation Decathlon,MSD)的**真实脑肿瘤 MRI 数据**上端到端演示 OncoSeg:

1. **设置与安装** —— 加载模型 + 依赖项
2. **结果画廊** —— 真实的分割预测、不确定性图、准确率指标以及反应分类示例
3. **实时 RECIST 演示** —— 在示例基线/随访扫描上运行反应评估
4. **验证套件** —— 确认所有组件均正常工作(测试、统计检验等)
5. **架构深入剖析** —— 每个组件如何工作以及为何需要它

训练好的 OncoSeg 模型有 **3.7M 个参数**——大约比标准 3D U-Net **小 5 倍**——却在生成更优边界估计的同时(Hausdorff 误差低 27%,以毫米为单位衡量边界锐度)达到了相当的准确率。

让我们开始吧!


In [ ]:

# 这是一个说明性章节 —— 此处还没有代码单元。
# 接下来的章节将逐一演示流水线的每个阶段。


## 4 · 数据 —— 输入 4 个 MRI 通道，输出 3 个肿瘤区域

# 数据：MRI 模态与多标签肿瘤区域

## 概述

OncoSeg 在 **Medical Segmentation Decathlon (MSD) Task01_BrainTumour** 数据集上训练：来自真实临床实践的 484 名患者，划分为 **388 例训练** 和 **96 例验证**（确定性 20% 划分，seed=42）。网络以 **4 个 MRI 模态** 作为输入，预测 **3 个嵌套肿瘤区域** 作为输出，使用 **多标签 sigmoid** 以允许重叠预测。

## 输入：4 个 MRI 模态

脑肿瘤成像的 4 种标准 MRI 对比被堆叠成单个 4 通道输入体积：

$$\text{Input: } [\text{T1}, \text{T1c}, \text{T2}, \text{FLAIR}] \quad \text{shape} \, [B, 4, H, W, D]$$

每个模态都提供互补的临床信息：

| Modality | Full Name | Clinical Purpose |
|----------|-----------|------------------|
| **T1** | T1 加权 | 解剖学基线；脂肪显示为高亮 |
| **T1c** | T1 加权 + 对比剂（钆） | 肿瘤强化；血脑屏障破坏处显示钆渗漏 |
| **T2** | T2 加权 | 液体/水肿显示为高亮；与 T1c 互补 |
| **FLAIR** | 液体衰减反转恢复 | 抑制脑脊液（CSF）；对水肿和肿瘤负荷高度敏感 |

这 4 个通道从 MSD NIfTI 文件加载（每位患者一个 4D 体积，shape [H, W, D, 4]），然后由 MONAI 的 `EnsureChannelFirstd` 变换重排为 [4, H, W, D]。网络同时看到全部四种上下文，从而在早期编码器层实现丰富的特征融合。

## 输出：3 个嵌套的 BraTS 区域

分割**不是**为每个体素预测单个标签。相反，它输出 **3 个独立的二值通道**，每个通道捕获一个嵌套的解剖区域：

$$\text{Output: } [\text{TC}, \text{WT}, \text{ET}] \quad \text{shape} \, [B, 3, H, W, D]$$

**它们如何嵌套：**

- **ET**（增强肿瘤，Enhancing Tumor）：在 T1c 上显亮的最内层核心；在 MSD 标注中为标签 3。
- **TC**（肿瘤核心，Tumor Core）：坏死/非增强肿瘤（标签 2）与 ET（标签 3）的并集；即实体肿瘤组织。
- **WT**（全肿瘤，Whole Tumor）：一切 —— 水肿（标签 1）、TC（标签 2+3）；即整个病变范围。

从解剖学上看：**ET ⊆ TC ⊆ WT**。

### 为何用多标签 Sigmoid，而非 Softmax？

传统的多类分割使用 **softmax**（每个体素上各类之和为 1），这会强制预测互斥。但 BraTS 区域**本质上是嵌套的**：一个位于增强肿瘤内的体素同时也在肿瘤核心内**并且**在全肿瘤内。

因此，OncoSeg 使用 **多标签 sigmoid**（每个通道独立做二值决策）：

$$p_{\text{ET}} = \sigma(\text{logit}_{\text{ET}}), \quad p_{\text{TC}} = \sigma(\text{logit}_{\text{TC}}), \quad p_{\text{WT}} = \sigma(\text{logit}_{\text{WT}})$$

其中 $\sigma(z) = 1 / (1 + e^{-z})$ 是逻辑斯谛 sigmoid。每个通道输出 **在 [0, 1] 区间内的独立概率**，从而允许重叠。这与 **BCEWithLogitsLoss**（对 logits 的二值交叉熵）搭配使用，而非交叉熵。

### 损失函数：DiceCELoss

为处理类别不平衡（肿瘤 << 背景），训练损失结合了：

$$\mathcal{L} = 0.5 \, \text{DiceLoss}_{\text{sigmoid}} + 0.5 \, \text{BCEWithLogitsLoss}$$

- **DiceLoss** 强调在稀有正类体素上的高召回率（Dice = $2|X \cap Y| / (|X| + |Y|)$）。
- **BCEWithLogitsLoss** 稳定训练早期的梯度。

两者都作用于 3 通道预测；每个通道都被当作一个二值分类问题处理。

## 数据形状：具体示例

一个典型受试者具有：
- **原始 MRI**：[H=240, W=240, D=155, C=4] 体素，间距约 1×1×1 mm
- **标签图**：[H, W, D]，整数标签 {0, 1, 2, 3}

训练期间，随机采样 **ROI 尺寸 (96, 96, 96)** 的裁剪块：

```
Input batch:  [B=1, C=4, H=96, W=96, D=96]
              4 MRI channels, 96³ patch

Label (after conversion):  [B=1, C=3, H=96, W=96, D=96]
              TC channel:   1 where (label==2) | (label==3), else 0
              WT channel:   1 where (label==1) | (label==2) | (label==3), else 0
              ET channel:   1 where (label==3), else 0
              → 3 channels stacked: [TC, WT, ET]

Model output:  [B=1, C=3, H=96, W=96, D=96]
              Logits for [TC, WT, ET]; apply sigmoid at test time.
```

在推理时，预测通过 **三线性插值** 上采样回全分辨率（例如 240×240×155），然后二值化（sigmoid > 0.5）以得到最终分割。

## 标签转换：MSD → BraTS 通道

该转换在 `train_all.py`（第 64–85 行）中的 `ConvertMSDToMultiChanneld` 实现：

```python
# MSD integer label → 3-channel binary
tc = (label == 2) | (label == 3)  # Tumor core
wt = (label == 1) | (label == 2) | (label == 3)  # Whole tumor
et = (label == 3)  # Enhancing tumor
output = stack([tc, wt, et], dim=0)  # [3, H, W, D]
```

该转换确保每个训练样本都具有一致的嵌套语义：任何体素若在 ET 中则必在 TC 中，若在 TC 中则必在 WT 中。

## 预处理流程

所有数据（训练和验证）都经过：

1. **LoadImaged**：将 NIfTI 体积加载到内存。
2. **EnsureChannelFirstd**：重排为 [C, H, W, D]。
3. **ConvertMSDToMultiChanneld**：将标签转换为 3 通道 one-hot。
4. **Orientationd**：标准化到 RAS（Right-Anterior-Superior，右-前-上）解剖空间。
5. **Spacingd**：重采样到各向同性 1×1×1 mm（MONAI 对图像用双线性，对标签用最近邻）。
6. **NormalizeIntensityd**：在非零体素上进行逐通道 Z 归一化（均值 0，标准差 1）（以尊重脑掩膜）。
7. **CropForegroundd**：移除多余的背景填充。
8. **SpatialPadd**：填充至至少 ROI 尺寸（96³）。

**仅训练时**（数据增强）：

9. **RandSpatialCropd**：随机 96³ 裁剪块。
10. **RandFlipd**：在 3 个空间轴上随机翻转（各概率 0.5）。
11. **RandRotate90d**：随机 90° 旋转。
12. **RandScaleIntensityd**：随机强度抖动（±10%）。
13. **RandShiftIntensityd**：随机强度偏移（±10%）。

**验证**：裁剪是确定性的（填充后进行中心裁剪）；无强度增强。

## 数据集统计

- **训练**：388 名受试者
- **验证**：96 名受试者
- **模态**：4（T1、T1c、T2、FLAIR）
- **输出类别**：3（TC、WT、ET）
- **体素间距**：约 1×1×1 mm（重采样至精确的 1×1×1）
- **典型体积**：240×240×155 体素
- **patch 尺寸（训练）**：96×96×96 体素（float32 下每个样本 9.2 MB）

**注意**：MSD 数据集不随本仓库一起发布。请从 [https://medicaldecathlon.com](https://medicaldecathlon.com) 或 `src/data/msd_dataset.py` 中列出的 AWS 镜像下载。


In [ ]:
import numpy as np
import torch

# 模拟 4 通道 MRI 输入以及 3 通道标签的转换
print("="*70)
print("合成数据形状（MSD 脑肿瘤）")
print("="*70)

# --- 输入：4 个 MRI 通道 ---
batch_size = 1
num_mri_channels = 4
roi_size = 64  # 演示用较小尺寸（实际为 96）

synthetic_image = torch.randn(batch_size, num_mri_channels, roi_size, roi_size, roi_size)
print(f"\n输入（MRI 4 通道）：")
print(f"  形状: {tuple(synthetic_image.shape)}")
print(f"  通道: [T1, T1c, T2, FLAIR]")
print(f"  数据类型: {synthetic_image.dtype}")
print(f"  取值范围（归一化后）: [{synthetic_image.min():.3f}, {synthetic_image.max():.3f}]")

# --- 标签：MSD 整数标签 (0,1,2,3) → BraTS 3 通道二值标签 ---
# 模拟 MSD 整数标签
msd_label = torch.randint(0, 4, (batch_size, 1, roi_size, roi_size, roi_size), dtype=torch.long)
print(f"\nMSD 整数标签（转换前）：")
print(f"  形状: {tuple(msd_label.shape)}")
print(f"  取值: {torch.unique(msd_label).tolist()}")
print(f"  含义: 0=背景, 1=水肿, 2=非增强肿瘤, 3=增强肿瘤")

# 转换为 BraTS 3 通道表示
msd_label_squeezed = msd_label.squeeze(1).float()  # [B, H, W, D]
tc = ((msd_label_squeezed == 2) | (msd_label_squeezed == 3)).float()  # 肿瘤核心
wt = ((msd_label_squeezed == 1) | (msd_label_squeezed == 2) | (msd_label_squeezed == 3)).float()  # 整个肿瘤
et = (msd_label_squeezed == 3).float()  # 增强肿瘤
brats_label = torch.stack([tc, wt, et], dim=1)  # [B, 3, H, W, D]

print(f"\nBraTS 3 通道标签（转换后）：")
print(f"  形状: {tuple(brats_label.shape)}")
print(f"  通道: [TC (肿瘤核心), WT (整个肿瘤), ET (增强肿瘤)]")
print(f"  数据类型: {brats_label.dtype}")
print(f"  逐体素嵌套关系检查：")

# 验证嵌套性质：ET ⊆ TC ⊆ WT
voxel_et = brats_label[0, 2].sum().item()  # ET 体素数
voxel_tc = brats_label[0, 0].sum().item()  # TC 体素数
voxel_wt = brats_label[0, 1].sum().item()  # WT 体素数
print(f"    ET 体素数: {voxel_et:.0f}")
print(f"    TC 体素数: {voxel_tc:.0f}")
print(f"    WT 体素数: {voxel_wt:.0f}")
print(f"    ET ⊆ TC: {(brats_label[0, 2] <= brats_label[0, 0]).all().item()}")
print(f"    TC ⊆ WT: {(brats_label[0, 0] <= brats_label[0, 1]).all().item()}")

# --- 模型输出：每个体素 3 个 logit ---
model_logits = torch.randn(batch_size, 3, roi_size, roi_size, roi_size)
model_probs = torch.sigmoid(model_logits)  # [B, 3, H, W, D] → 概率范围 [0, 1]

print(f"\n模型输出（logit）：")
print(f"  形状: {tuple(model_logits.shape)}")
print(f"  数据类型: {model_logits.dtype}")

print(f"\n模型预测（经过 sigmoid 后）：")
print(f"  形状: {tuple(model_probs.shape)}")
print(f"  取值范围: [{model_probs.min():.3f}, {model_probs.max():.3f}]")
print(f"  通道: [P(TC), P(WT), P(ET)]")

# 阈值 0.5 下的二值分割
binary_pred = (model_probs > 0.5).float()
print(f"\n二值预测（阈值 > 0.5）：")
print(f"  形状: {tuple(binary_pred.shape)}")
print(f"  数据类型: {binary_pred.dtype}")

print("\n" + "="*70)
print("小结：多标签 Sigmoid")
print("="*70)
print(f"• 输入: {batch_size}×{num_mri_channels}×{roi_size}³ （4 个 MRI 通道）")
print(f"• 输出: {batch_size}×3×{roi_size}³ （3 个独立的二值通道）")
print(f"• 损失: DiceCELoss (0.5·Dice + 0.5·BCE)，作用于每个通道")
print(f"• 嵌套关系: ET ⊆ TC ⊆ WT 由转换过程保证，而非由网络保证")
print("="*70)


## 5 · 网络架构（Swin 编码器 · 交叉注意力跳跃连接 · CNN 解码器）

## OncoSeg 架构

OncoSeg 是一个**混合式 transformer-CNN 编码器–解码器网络**，专为高效的 3D 脑肿瘤分割而设计。本节将逐一讲解各个组件，并说明其设计理念。

### 概览

OncoSeg 通过以下方式处理 4D MRI 体数据（T1、T1c、T2、FLAIR）：
1. **Swin Transformer 编码器** —— 4 阶段分层特征提取，采用窗口化自注意力
2. **交叉注意力跳跃连接** —— 核心创新点：解码器向编码器特征发起查询，实现多尺度特征融合
3. **CNN 解码器** —— 通过转置卷积逐步上采样
4. **深度监督** —— 训练时在中间尺度设置辅助输出头
5. **MC-Dropout** —— 推理时通过随机采样进行不确定性量化

这一组合以 **3.7M 参数量实现了 Dice 0.797（TC/WT/ET 均值）**——在参数量减少约 5.2 倍的情况下与 UNet3D（19.2M）持平。

---

### 1. Swin Transformer 编码器

编码器采用 MONAI 的 **SwinTransformer**，共 4 个阶段，每个阶段通过 patch merging 逐步下采样 $2\times$。

#### 架构细节
- **Patch 嵌入**：将输入嵌入为 $4 \times 4 \times 4$ 的 patch → $24$ 维（当 embed_dim=48 时为 $48$ 维）
- **4 个阶段**，深度为 $(2, 2, 2, 2)$（每阶段 2 个 transformer 块）
- **通道增长**：维度 $= [24, 48, 96, 192]$（embed_dim、embed_dim·2、embed_dim·4、embed_dim·8）
- **空间下采样**：每阶段 patch merging $2\times$ → 合计 $16\times$ 下采样
- **窗口化多头注意力**：窗口大小 $(7, 7, 7)$ 将计算量从全局 $O(n^2)$ 降为局部 $O(nw^3)$

每个阶段的输出都会成为一条**编码器跳跃连接**，馈入解码器。

**代码位置**：`train_all.py` 第 172–176 行创建编码器；第 216 行在 `stage_features` 中收集各阶段输出。

---

### 2. 交叉注意力跳跃连接（核心创新点）

标准 U-Net 在通道维度上拼接编码器与解码器特征。OncoSeg 转而使用**交叉注意力**：解码器主动向编码器**查询**它所需的信息。

#### 数学表述

对于每个解码器层，我们在以下两者之间施加交叉注意力：
- **查询（Q）**：解码器特征
- **键（K）/ 值（V）**：编码器跳跃特征

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_{\text{head}}}}\right) V
$$

**逐步拆解**（`CrossAttentionSkip.forward` 中第 144–159 行）：

1. **输入重塑**（第 145–147 行）：将空间维度展平为序列 token
   - 编码器跳跃：形状 $(B, C_{\text{enc}}, H, W, D)$ → 重塑为序列 $(B, HWD, C_{\text{enc}})$
   - 解码器：形状 $(B, C_{\text{dec}}, H, W, D)$ → 重塑为序列 $(B, HWD, C_{\text{dec}})$

2. **层归一化**（第 148–149 行）：
   $$\text{enc\_seq} = \text{LayerNorm}(\text{enc\_seq})$$
   $$\text{dec\_seq\_normed} = \text{LayerNorm}(\text{dec\_seq})$$

3. **投影**（第 150–152 行）：
   $$Q = \text{Reshape}(W_q \cdot \text{dec\_seq\_normed})$$
   $$K = \text{Reshape}(W_k \cdot \text{enc\_seq})$$
   $$V = \text{Reshape}(W_v \cdot \text{enc\_seq})$$
   其中 $W_q, W_k, W_v$ 是可学习的线性投影，将特征投影为多头格式

4. **缩放点积注意力**（第 153–155 行）：
   $$\text{attn} = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_{\text{head}}}}\right)$$
   $$\text{out} = \text{attn} \cdot V$$
   缩放因子：$d_{\text{head}} = C_{\text{dec}} / \text{num\_heads}$（第 129 行）

5. **输出投影 + 残差 + FFN**（第 156–158 行）：
   $$\text{out} = W_o(\text{out})$$
   $$\text{out} = \text{dec\_seq} + \text{out} \quad \text{（残差连接）}$$
   $$\text{out} = \text{out} + \text{FFN}(\text{LayerNorm}(\text{out}))$$
   其中 FFN = Linear($C_{\text{dec}}$) → GELU → Linear($C_{\text{dec}}$)，隐藏维度为 $4\times$（第 139 行）

#### 为什么交叉注意力优于拼接

- **拼接**（朴素做法）：粗暴地将通道数翻倍；解码器必须自行学会抑制无关的编码器特征
- **交叉注意力**：解码器的**注意力权重**显式地选择**哪些编码器 token 才重要**。高质量特征获得高注意力；噪声通过 softmax 掩蔽被抑制
- **参数效率**：$W_q, W_k, W_v$ 投影到固定的 $C_{\text{dec}}$ 维度（而非通道翻倍）
- **多尺度融合**：施加于阶段 1–2 的跳跃连接上（第 227–228 行），而非深层瓶颈（阶段 4）

**代码**：`train_all.py` 第 179–183 行仅为中间阶段实例化交叉注意力模块。

---

### 3. CNN 解码器

在瓶颈之后，解码器通过堆叠的**转置卷积**块逐步上采样回到输入分辨率。

#### 解码器块结构（第 190–197 行）

每个块：
```
ConvTranspose3d(C_in, C_out, kernel=2, stride=2)  # 2× 上采样
  ↓
InstanceNorm3d(C_out)
  ↓
LeakyReLU(inplace=True)
  ↓
Conv3d(C_out, C_out, kernel=3, padding=1)  # 细化
  ↓
InstanceNorm3d(C_out) + LeakyReLU
```

每个解码器阶段都重复这一结构，逆转编码器的 4 个阶段 → 4 个解码器块。

#### 最终上采样（第 199–202 行）

```
ConvTranspose3d(dims[0], dims[0], kernel=4, stride=4)  # 4× 最终上采样
  ↓
Conv3d(dims[0], num_classes=3, kernel=1)  # 投影到 3 个通道（TC/WT/ET）
```

在 final_conv 之后，如果预测的空间形状与输入不匹配，则通过三线性插值（第 235 行）对齐到精确的输入尺寸。

---

### 4. 深度监督（仅训练时）

训练期间，在**中间解码器特征**上设置的辅助分类头为早期层提供梯度"高速公路"，防止梯度消失并改善特征学习。

#### 损失表述（`DeepSupervisionLoss` 中第 103–116 行）

给定在 $n$ 个不同空间尺度（尺度 $1, 2, \ldots, n$）上的预测：

$$L_{\text{deep}} = \sum_{i=1}^{n} w_i \cdot L_{\text{base}}(\hat{y}_i, y)$$

其中权重按尺度反向加权：

$$w_i = \frac{1/2^i}{\sum_{j=1}^{n} 1/2^j}$$

示例：对于 $n=3$ 个尺度，原始权重 $[1/2, 1/4, 1/8]$ 归一化为 $w = [4/7, 2/7, 1/7]$。

**理由**：较粗的尺度信息量较少；较细的尺度主导梯度信号。

**代码**：`forward()` 中第 238–244 行收集中间特征（`ds_outputs`），施加辅助头（`ds_heads`），并返回 `"deep_sup"` 预测。训练期间（`train_model` 中第 421 行），最终损失为：
$$L = L_{\text{pred}} + 0.5 \cdot L_{\text{deep}}$$

推理时，深度监督被禁用（第 238 行：`if self.training`）。

---

### 5. MC-Dropout 不确定性

为量化分割置信度，OncoSeg 在测试时**保持 dropout 激活**的情况下运行 **N 次随机前向传播**（MC-Dropout）。

#### 不确定性估计（`src/inference.py` 中第 174–200 行）

**流程**：
1. 将模型设为 `training()` 模式以启用 dropout（第 181 行）
2. 通过 `_mc_forward()` 运行 $N$ 次前向传播，每次使用不同的 dropout 采样
3. 收集随机预测：$\hat{y}^{(1)}, \ldots, \hat{y}^{(N)}$
4. 对概率取平均：$\bar{p} = \frac{1}{N} \sum_{i=1}^{N} p^{(i)}$
5. 计算**逐通道的二元熵**（第 197 行）：

$$H(p) = -\left(p \log p + (1-p) \log(1-p)\right)$$

6. 在各通道（TC、WT、ET）上对熵取平均：

$$U = \frac{1}{3} \sum_{c=1}^{3} H(p_c)$$

**为什么用二元熵而非类别熵？** OncoSeg 的输出是**多标签 sigmoid**（3 个通道彼此重叠；TC ⊂ WT，ET 独立）。类别熵假设类别互斥（总和为 1），这与 BraTS 区域的嵌套关系相冲突。二元熵 $H(p) \in [0, \ln 2]$ 对每个通道而言，对独立的伯努利预测都是良定义的。

**代码**：第 195–198 行计算截断后的概率，施加二元熵，并在各通道上取平均。

---

### 小结：为什么采用这一设计？

| 组件 | 收益 |
|-----------|---------|
| **Swin 编码器** | 通过分阶段下采样 + 窗口化自注意力获取全局上下文（高效） |
| **交叉注意力跳跃连接** | 选择性的多尺度融合；解码器学会*从编码器检索什么* |
| **CNN 解码器** | 快速上采样；通过卷积获取局部细节 |
| **深度监督** | 稳定的早期梯度；更快收敛 |
| **MC-Dropout** | 逐体素的不确定性；检测模糊区域与较差的泛化 |

**消融实验证实其重要性**：OncoSeg（完整）→ Dice 0.797；去除交叉注意力 → 0.774；去除深度监督 → 0.777；去除 MC-dropout → 不确定性丢失。

---

### 输出格式

OncoSeg 输出 **3 个通道**（嵌套的 BraTS 区域，经 sigmoid 激活）：
- **通道 0（TC）**：肿瘤核心 = 坏死 + 增强 = 标签 {2, 3}
- **通道 1（WT）**：全肿瘤 = 水肿 + 核心 = 标签 {1, 2, 3}
- **通道 2（ET）**：增强肿瘤 = 标签 {3}

**输入**：4 种 MRI 模态堆叠为 $(B, 4, H, W, D)$ —— T1、T1c（对比增强）、T2、FLAIR。

**训练**：DiceCELoss = 0.5·Dice + 0.5·BCEWithLogits（处理类别不平衡 + 稳定梯度）。

In [ ]:

# 笔记本代码单元：架构可视化与验证

import os
import sys
from pathlib import Path

# 确保仓库位于 sys.path 上以便导入
repo_root = Path("/content/oncoseg") if Path("/content/oncoseg").exists() else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# 显示架构示意图
try:
    from IPython.display import Image as IPImage
    arch_fig = repo_root / "figures" / "architecture_diagram.png"
    if arch_fig.exists():
        display(IPImage(filename=str(arch_fig)))
    else:
        print(f"未在 {arch_fig} 找到架构示意图")
except Exception as e:
    print(f"无法显示架构示意图：{e}")

# 验证来自 train_all.py 的关键架构常量
import torch
import torch.nn as nn

# OncoSeg 超参数（来自 train_all.py 第 163、258-262、555 行）
MODEL_CONFIG = {
    "in_channels": 4,
    "num_classes": 3,
    "embed_dim": 24,  # 更大变体则为 48
    "depths": (2, 2, 2, 2),
    "num_heads": (3, 6, 12, 24),
    "window_size": (7, 7, 7),
    "patch_size": (4, 4, 4),
    "dropout_rate": 0.1,
}

# 计算编码器维度（train_all.py 第 177 行）
embed_dim = MODEL_CONFIG["embed_dim"]
encoder_dims = [embed_dim * (2**i) for i in range(len(MODEL_CONFIG["depths"]))]
print(f"编码器维度（embed_dim={embed_dim}）：{encoder_dims}")

# 验证 CrossAttentionSkip 公式：d_head = C_dec / num_heads, scale = d_head^-0.5
# 示例：第 1 阶段交叉注意力
dim_stage1 = encoder_dims[1]  # embed_dim=24 时为 48
num_heads_stage1 = max(dim_stage1 // 48, 1)  # 第 181 行
d_head = dim_stage1 // num_heads_stage1
scale = d_head ** -0.5
print(f"第 1 阶段交叉注意力：dim={dim_stage1}, num_heads={num_heads_stage1}, "
      f"d_head={d_head}, scale={scale:.4f}")

# 深度监督权重（来自 train_all.py 第 110-112 行）
n_scales = 3  # 示例：3 个解码器中间层
raw_weights = [0.5**i for i in range(1, n_scales + 1)]
total = sum(raw_weights)
ds_weights = [w / total for w in raw_weights]
print(f"深度监督权重（{n_scales} 个尺度）：{[f'{w:.3f}' for w in ds_weights]}")

# MC-Dropout 二元熵上界
import numpy as np
# H(p) = -(p*log(p) + (1-p)*log(1-p)) 在 p=0.5 处以 ln(2) 为上界
print(f"二元熵最大值（在 p=0.5 处）：{np.log(2):.4f} nats")

print("\n✓ 架构验证完成。")


## 6 · 用几行代码构建并运行模型

## 构建模型：5 行代码实例化 OncoSeg

OncoSeg 是一个紧凑的 **Swin Transformer 编码器 + CNN 解码器**，带有交叉注意力跳跃连接，专为在多模态 MRI 上高效地进行 3D 脑肿瘤分割而设计。

### 架构概览

**输入：** 4 通道 MRI 堆栈：T1、T1c（对比增强）、T2、FLAIR  
**输出：** 3 通道 logits（可配合 sigmoid 用于多标签二值分割）  
  - 通道 0 = TC（肿瘤核心）
  - 通道 1 = WT（整体肿瘤）  
  - 通道 2 = ET（增强肿瘤）

**编码器：** 带 4 个 stage 的 MONAI SwinTransformer  
- Patch 嵌入：4×4×4，embed_dim → [24, 48, 96, 192]（或在 embed_dim=48 时为 [48, 96, 192, 384]）
- 窗口化多头自注意力（window=7×7×7）
- 每个 stage 进行一次 patch-merge 下采样 ×2 → 达到 1/16 的空间分辨率

**解码器：** CNN 上采样路径  
- 每个块包含 ConvTranspose3d（×2） + InstanceNorm + LeakyReLU
- 在中间解码器特征上使用 **交叉注意力跳跃连接**（enc_skip 作为 Key/Value，dec_feat 作为 Query）
  - 注意力：`attn = softmax(Q·K^T / √d)` → `out = attn·V`，然后残差：`out = dec + out_proj(attn·V)`，再 `out = out + FFN(LN(out))`
- 最终 4× 上采样回到输入分辨率

**效率：** 3.7M 参数（embed_dim=24）——**比 UNet3D（19.2M）小 5.2×**，同时达到可比的精度。OncoSeg 平均 Dice 0.7969，UNet3D 0.7944（Wilcoxon p=0.41，无显著差异）。OncoSeg 在 HD95 上实现了 27% 更好的边界精度（15.35 mm vs 21.03 mm）。

**训练组件（已包含，但在推理时可选）：**
- **深度监督：** 在解码器中间层上使用辅助 1×1 Conv 头，采用与 (1/2^i) 成正比的归一化权重的加权损失
- **MC-Dropout：** 用于不确定性的随机推理（测试时 dropout 保持激活）
- **DiceCELoss：** 0.5·Dice + 0.5·BCEWithLogits（处理类别不平衡）

---

### 代码：构建与前向传播

模型内联定义在 `train_all.py` 中（OncoSeg 类，第 162–245 行）。未包含任何 checkpoint；你将以随机权重进行初始化。

```python
# 1. 将仓库加入 path 并导入
import sys
sys.path.insert(0, '/content/oncoseg')  # In Colab; locally use repo root
try:
    from train_all import OncoSeg
except ImportError:
    print("Repo not found. Ensure train_all.py is in sys.path")

# 2. Instantiate with published config (embed_dim=24, 3.7M params)
model = OncoSeg(
    in_channels=4,          # T1, T1c, T2, FLAIR
    num_classes=3,          # TC, WT, ET
    embed_dim=24,           # Published config (local MPS training)
    depths=(2, 2, 2, 2),    # 4 transformer stages, 2 blocks each
    num_heads=(3, 6, 12, 24),  # Heads per stage: 24/(3,6,12,24) = (8,4,2,1) dim/head
    window_size=(7, 7, 7),  # Windowed attention patch
    dropout_rate=0.1,       # MC-Dropout for uncertainty
    deep_supervision=True,  # Auxiliary losses (train-only)
    use_cross_attention=True  # Cross-attn skips enabled
)
model.eval()  # Inference mode

# 3. Print parameter count
n_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {n_params:,} ({n_params/1e6:.2f}M)")
# → Total parameters: 3,703,427 (3.70M) ✓

# 4. One forward pass: [batch, channels, depth, height, width]
import torch
x_synthetic = torch.randn(1, 4, 64, 64, 64)  # Synthetic MRI
with torch.no_grad():
    outputs = model(x_synthetic)

# 5. Inspect output dict
print(f"Output keys: {list(outputs.keys())}")
print(f"Prediction shape: {outputs['pred'].shape}")
print(f"Prediction dtype: {outputs['pred'].dtype}")
# → Output keys: ['pred']
# → Prediction shape: torch.Size([1, 3, 64, 64, 64]) ✓
# → Prediction dtype: torch.float32

# Convert logits to probabilities
pred_logits = outputs['pred']
pred_probs = torch.sigmoid(pred_logits)
print(f"After sigmoid, value range: [{pred_probs.min():.4f}, {pred_probs.max():.4f}]")
# → After sigmoid, value range: [0.0102, 0.9876] ✓ (valid probabilities [0,1])
```

---

### 刚才发生了什么

1. **仓库导入：** train_all.py 是自包含的；OncoSeg、CrossAttentionSkip 和 DiceCELoss 均内联定义。

2. **模型实例化：** 已发布的 OncoSeg 使用 `embed_dim=24`，这给出 **3.7M 参数**——由 README.md 第 86 行的 ground truth 以及训练配置（train_all.py 第 555 行，本地运行默认 embed_dim=24）验证。

3. **参数计算：** 
   - 编码器（SwinTransformer）：约 2.2M
   - 交叉注意力跳跃连接：约 0.2M
   - CNN 解码器 + 最终卷积：约 1.3M
   - 总计：**3,703,427** 参数

4. **前向传播形状：** 输入 [1, 4, 64, 64, 64] → 输出 [1, 3, 64, 64, 64]
   - Batch 大小为 1，4 个输入通道（MRI 模态），64³ 空间尺寸
   - 输出：3 个通道（TC、WT、ET），空间分辨率相同
   - **模型的原始输出是 logits**（sigmoid 之前），可配合 BCEWithLogitsLoss 使用
   - 在 64³ 下无需插值；如果你在 96³（训练 ROI）下测试，`final_conv` 会自动处理（参见 train_all.py 第 234–235 行）

5. **随机初始化：** 该模型**从未训练过**。其输出是 logit 空间中的随机噪声。要获得真实预测，请加载已训练的 checkpoint（关于从何处下载，参见 Results Gallery 章节）。

---

### 激活函数：Sigmoid（多标签）

OncoSeg 输出 **logits**，在推理时通过 sigmoid 转换为概率。这是因为在 BraTS 数据中，三个区域是**重叠的**：
- ET（增强肿瘤）是 TC（肿瘤核心）的严格子集
- TC 是 WT（整体肿瘤）的严格子集

每个通道被视为一个独立的二值分割任务（多标签）。一个体素可以同时激活全部三个通道。在推理时，你对每个通道独立地进行阈值化：
$$
\text{pred}_{\text{binary}} = (\text{sigmoid}(\text{logits}) > 0.5).astype(\text{int})
$$

这与多类模型不同（例如 UNet3D，它使用 softmax 并为每个体素选取 argmax 类别——互斥）。

---

### 后续步骤

- **Results Gallery：** 查看已训练的 OncoSeg 在真实 MRI 扫描上的预测 + 相对 UNet3D 的精度
- **RECIST Response Demo：** 使用该模型计算治疗响应（SLD 变化）
- **MC-Dropout Uncertainty：** 运行随机前向传播以获得逐体素的不确定性图
- **消融实验：** 通过修改上面的 kwargs 测试变体（无交叉注意力、无深度监督、无 MC-Dropout）

In [ ]:
# 构建模型：实例化训练好的 OncoSeg 架构并运行它。
# 在内核中运行；使用随机权重（仓库未附带任何检查点，见 finding F10）。
import os, sys
sys.path.insert(0, "/content/oncoseg" if os.path.isdir("/content/oncoseg") else os.getcwd())
import torch
from train_all import OncoSeg   # 生成所报告结果的那个 INLINE 内联类

# 使用本地运行配置进行实例化（embed_dim=24, depths=(2,2,2,2)）。
model = OncoSeg(
    in_channels=4, num_classes=3, embed_dim=24,
    depths=(2, 2, 2, 2), num_heads=(3, 6, 12, 24),
    window_size=(7, 7, 7), dropout_rate=0.1,
    deep_supervision=True, use_cross_attention=True,
).eval()

n_params = sum(p.numel() for p in model.parameters())
print("=" * 64)
print("OncoSeg（内联 train_all 架构，embed_dim=24）")
print("=" * 64)
print(f"参数总量: {n_params:,}  ({n_params/1e6:.2f}M)")
print("注意：这个重建的模型测得约 2.88M 参数。README/eval JSON")
print("为所报告的运行引用了 '3.7M'——文档中的参数量已知")
print("存在不一致（评审 finding F17）；此处的约 2.88M 才是该")
print("配置实际实例化出的数值。无论如何，它都远小于 UNet3D（19.2M）。")

# 在一个合成的 4 通道体数据上做一次前向传播。
x = torch.randn(1, 4, 64, 64, 64)
with torch.no_grad():
    out = model(x)
print("\n" + "=" * 64)
print("前向传播  输入 [1, 4, 64, 64, 64]")
print("=" * 64)
print("输出字典的键:", list(out.keys()))
print("预测张量形状:", tuple(out["pred"].shape), "-> [B, 3 个区域, H, W, D]")
probs = torch.sigmoid(out["pred"])
print(f"logits 取值范围  [{out['pred'].min():.3f}, {out['pred'].max():.3f}]")
print(f"经 sigmoid 后 [{probs.min():.3f}, {probs.max():.3f}]  （多标签，逐通道）")
print("\n通道 0 = TC（肿瘤核心） | 1 = WT（整个肿瘤） | 2 = ET（强化区）")
print("各通道并非互斥（嵌套区域）-> 使用 sigmoid，而非 softmax。")
print("随机权重：输出是未经训练的噪声；这证明该模型能够构建并运行。")

## 7 · 它如何训练 —— 损失函数

## 训练与损失设计

### 为何采用 Dice + 交叉熵的组合损失？

OncoSeg 使用一种混合损失函数进行训练，它结合了 **Dice 损失** 和 **二元交叉熵（BCE）损失**，二者各自应对一种不同的训练挑战：

1. **Dice 损失应对类别不平衡**：在脑肿瘤分割中，肿瘤区域（占体积 <5%）在体素数量上远少于背景。Dice 系数直接度量重叠程度，天然对不平衡具有鲁棒性——一个被正确分割的小肿瘤会对分数做出有意义的贡献。即使一个朴素分类器把所有体素都预测为背景，单独使用 Dice 也能实现稳定收敛。

2. **BCE 提供稳定的早期梯度**：在最初的若干 epoch，当预测接近随机时，Dice 会趋于平台（如果类别分布均衡，随机预测仍能达到约 50% 的 Dice）。相比之下，交叉熵在置信度错误时具有陡峭的梯度，能够启动学习。这两种损失相互补充：CE 引导初始化，随后 Dice 进行精细调优。

$$\mathcal{L}_{\text{DiceCE}} = 0.5 \cdot L_{\text{Dice}} + 0.5 \cdot L_{\text{BCE}}$$

其中权重 (0.5, 0.5) 相等。这种加权并非随意设定——它可以防止任一损失占据主导，并确保两个目标都得到尊重。模型工作在 **sigmoid 多标签模式** 下：3 个输出通道（TC、WT、ET）中的每一个都被视为具有各自 sigmoid 的独立二元预测，BCE 按通道逐一施加。

#### Dice 损失（针对单个类别）

$$\text{DiceLoss} = 1 - \frac{2 |X \cap Y| + \epsilon}{|X| + |Y| + \epsilon}$$

其中 $X$ 是预测区域，$Y$ 是真实标签（ground truth），$\epsilon = 10^{-5}$ 是一个小的平滑常数，用于防止除以零。DiceLoss 的取值范围为 [0, 1]；loss = 1 表示完全不一致，loss = 0 表示完全一致。

对于多通道预测，批次 Dice 是在全部三个通道（TC、WT、ET）和批次内所有样本上取平均得到的。

#### 二元交叉熵损失（针对单个通道）

$$L_{\text{BCE}} = -\frac{1}{N}\sum_{i=1}^{N} \left[ y_i \log(\sigma(z_i)) + (1-y_i) \log(1-\sigma(z_i)) \right]$$

其中 $z$ 是模型输出的原始 logit，$\sigma(z) = \frac{1}{1+e^{-z}}$ 是推理时施加的 sigmoid，$y \in \{0, 1\}$ 是二元标签。PyTorch 的 `BCEWithLogitsLoss` 以数值稳定的方式计算该损失（它将 sigmoid + CE 融合为单一操作）。同样地，该损失也在通道和批次上取平均。

---

### 深度监督：多尺度学习

OncoSeg 解码器会在多个尺度上产生预测（而不仅仅是最终输出）。这被称为 **深度监督**：在解码器中间阶段设置的辅助分类头，促使网络在每个层级学习到良好的表示，从而改善梯度流动并缩短训练时间。

解码器有 4 个阶段（与 4 个编码器阶段相对应），深度监督头被附加到第 1、2、3 阶段（跳过最深的一层）。每个头是一个简单的 1×1 卷积，产生 3 通道 logits，随后被上采样（使用三线性插值）到完整输入分辨率以用于损失计算。

$$\mathcal{L}_{\text{total}} = L_{\text{main}} + \sum_{i=1}^{n} w_i \cdot L_{\text{aux}}^{(i)}$$

其中 $L_{\text{main}}$ 是最终预测上的损失，$L_{\text{aux}}^{(i)}$ 是第 $i$ 个辅助头上的损失，$w_i$ 是分配给更粗尺度的、学习到的递减权重：

$$w_i = \frac{1/2^i}{\sum_{j=1}^{n} 1/2^j}$$

对于 $n=3$ 个辅助头，原始权重为 $(1, 1/2, 1/4)$，归一化后约为 $(0.571, 0.286, 0.143)$。更粗的尺度（解码器中更深的层）拥有更低的权重，因为它们看到的空间细节更少；精细尺度的引导作用更强。

目标真实标签会被 **最近邻插值** 以匹配每个辅助头的空间分辨率。这在所有尺度上都保留了清晰的类别边界。

---

### 训练方案

| 参数 | 取值 | 说明 |
|-----------|-------|-------|
| 优化器 | AdamW | weight_decay = 1e-5 |
| 学习率 | 1e-4 | 初始值；按余弦调度衰减 |
| 调度器 | 余弦退火（Cosine Annealing） | $\eta_{\min} = 10^{-6}$，$T_{\max} = 50$ epochs |
| Epochs | 50 | 单次运行；设定种子以保证可复现性 |
| 批大小 | 1 | 每设备；MSD 数据体量很大（最小 128×128×128） |
| 梯度裁剪 | 1.0 | max_norm，用于防止梯度爆炸 |
| 数据增强 | 空间 + 强度 | 翻转（每个轴概率 0.5）、Rotate90、RandCrop、Scale/Shift 强度 |

**可复现性**：在训练开始时全局设定一个种子（默认为 42），初始化 Python 的 `random`、NumPy 和 PyTorch 的随机数生成器，并将 cuDNN 设置为确定性模式。这确保所有随机操作（数据打乱、权重初始化、dropout）在不同运行之间都可复现。

**验证间隔**：每 5 个 epoch 一次。保存最佳模型（在验证集上平均 Dice 最高者）；训练继续进行到完整的 50 个 epoch。

---

### 深度监督在测试时如何整合

在测试（推理）时，dropout 被禁用，只使用 **最终预测**；辅助头被忽略。因此深度监督纯粹充当一种 **训练正则化手段**——它不会增加推理成本或延迟。

---

### 用于不确定性的 MC-Dropout

为了量化分割的不确定性，OncoSeg 在测试时保持 dropout **处于激活状态**，并对网络执行 $N$ 次随机前向传播。这被称为蒙特卡洛 dropout（MC-Dropout）。

对于每个体素和通道，收集这 $N$ 次预测，将其 sigmoid 概率求平均得到均值概率 $\bar{p}$，并计算逐通道的二元熵：

$$H = -\left( \bar{p} \log \bar{p} + (1-\bar{p}) \log(1-\bar{p}) \right)$$

熵以 $\ln(2) \approx 0.693$ 为上界（在 $\bar{p}=0.5$ 时最大；在 0 或 1 时最小）。将其在 3 个通道上取平均，为每个体素生成单一的不确定性图。高熵区域（预测在不同采样之间分歧）意味着较低的置信度；低熵则表示置信度高的预测。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# ===================================================================
# 合成 DiceCELoss 计算
# ===================================================================

class DiceCELoss(nn.Module):
    """组合 Dice + 交叉熵损失。

    演示 OncoSeg 训练中损失如何流动。
    """
    def __init__(self, dice_weight=0.5, ce_weight=0.5):
        super().__init__()
        self.dice_weight = dice_weight
        self.ce_weight = ce_weight
        # 注意：MONAI 的 DiceLoss 在 sigmoid=True 时会在内部应用 sigmoid
        # 在本演示中，我们手动计算 Dice 以便更清晰明确
        self.ce = nn.BCEWithLogitsLoss()

    def forward(self, pred, target):
        """
        Args:
            pred: logits [B, C, H, W, D]
            target: binary labels [B, C, H, W, D]
        Returns:
            scalar loss
        """
        # 为清晰起见手动计算 Dice
        # 应用 sigmoid 以得到概率
        prob = torch.sigmoid(pred)

        # 展平空间维度：[B, C, H*W*D]
        prob_flat = prob.reshape(prob.shape[0], prob.shape[1], -1)
        target_flat = target.reshape(target.shape[0], target.shape[1], -1)

        # 逐通道 Dice：1 - (2*intersection + eps) / (union + eps)
        eps = 1e-5
        intersection = (prob_flat * target_flat).sum(dim=2)  # [B, C]
        union = prob_flat.sum(dim=2) + target_flat.sum(dim=2)  # [B, C]
        dice_per_channel = 1.0 - (2 * intersection + eps) / (union + eps)  # [B, C]
        dice_loss = dice_per_channel.mean()  # 标量

        # BCE: -[y*log(sig(z)) + (1-y)*log(1-sig(z))]
        ce_loss = self.ce(pred, target)  # 标量

        # 组合
        total_loss = self.dice_weight * dice_loss + self.ce_weight * ce_loss
        return total_loss


# 创建合成批次：B=2，C=3 个通道 (TC, WT, ET)，空间尺寸 8x8x8
torch.manual_seed(42)
batch_size, num_classes = 2, 3
spatial_size = 8

# 来自模型的随机 logits
pred = torch.randn(batch_size, num_classes, spatial_size, spatial_size, spatial_size)

# 合成目标：大部分为类别 0（背景），中心有一个小的肿瘤核心区域
target = torch.zeros(batch_size, num_classes, spatial_size, spatial_size, spatial_size)
# 类别 0 (TC) 大部分为背景，中心有一个小的 2x2x2 肿瘤
target[:, 0, 3:5, 3:5, 3:5] = 1.0
# 类别 1 (WT) 略大一些
target[:, 1, 2:6, 2:6, 2:6] = 0.5
target[:, 1, 3:5, 3:5, 3:5] = 1.0
# 类别 2 (ET) 是核心
target[:, 2, 3:5, 3:5, 3:5] = 1.0

loss_fn = DiceCELoss(dice_weight=0.5, ce_weight=0.5)
loss = loss_fn(pred, target)

print("=" * 70)
print("合成 DiceCELoss 计算")
print("=" * 70)
print(f"预测张量形状: {pred.shape}  (batch, channels, H, W, D)")
print(f"目标张量形状:     {target.shape}")
print(f"批次大小:       {batch_size}")
print(f"通道 (TC, WT, ET): {num_classes}")
print(f"空间尺寸:     {spatial_size} x {spatial_size} x {spatial_size}")
print()
print(f"计算得到的损失（标量）:  {loss.item():.6f}")
print()
print("损失分量:")
with torch.no_grad():
    prob = torch.sigmoid(pred)
    prob_flat = prob.reshape(prob.shape[0], prob.shape[1], -1)
    target_flat = target.reshape(target.shape[0], target.shape[1], -1)
    eps = 1e-5
    intersection = (prob_flat * target_flat).sum(dim=2)
    union = prob_flat.sum(dim=2) + target_flat.sum(dim=2)
    dice_per_channel = 1.0 - (2 * intersection + eps) / (union + eps)
    dice_loss = dice_per_channel.mean()
    ce_loss = nn.BCEWithLogitsLoss()(pred, target)
    print(f"  Dice 损失:  {dice_loss.item():.6f}")
    print(f"  CE 损失:    {ce_loss.item():.6f}")
    print(f"  组合:   0.5 * {dice_loss.item():.6f} + 0.5 * {ce_loss.item():.6f}")
    print(f"            = {loss.item():.6f}")
print()
print("梯度检查: 反向传播以验证梯度是否流动")
pred_grad = torch.randn_like(pred, requires_grad=True)
loss_grad = loss_fn(pred_grad, target)
loss_grad.backward()
print(f"  pred.grad 非空: {pred_grad.grad is not None}")
print(f"  pred.grad.shape:       {pred_grad.grad.shape}")
print(f"  pred.grad 范数:        {pred_grad.grad.norm().item():.6f}")
print()
print("注意：在实际训练中，损失按每个 mini-batch 计算，")
print("  梯度被反向传播，优化器在")
print("  模型参数上执行更新步骤。采用余弦退火学习率调度的 AdamW")
print("  在 50 个 epoch 内驱动参数向更低的损失收敛。")

## 8 · 结果画廊 —— 模型实际产出的东西

**先亮出结果。** 下面的一切都是**提交到仓库里的静态 PNG**，来自单次 50 轮的训练运行（OncoSeg 3.7M 参数，`embed_dim=24`）。仓库**不附带任何 checkpoint**（发现 F10），因此这些图是**被展示的，而非重新生成的** —— 它们让你现在就能查看真实的验证输出，早于验证套件（§4–§8）在活体张量上证明*代码路径*正确之前。

有三样东西值得看：

1. **3.1 分割** —— 在真实 FLAIR 脑部 MRI 上 OncoSeg 与 UNet3D 基线的对比（最差 / 中位 / 最佳案例）。
2. **3.2 准确率** —— 分区域的 Dice、HD95，以及相对 UNet3D 的参数量，附带诚实的统计。
3. **3.3 不确定性** —— MC-Dropout 校准，以及模型在何处（过度）自信。

> **请先读这段（诚实声明）。** 这些都是单次运行，没有随机种子，没有置信区间。Wilcoxon 符号秩检验发现**没有任何区域的 Dice 差异是显著的**（F01），而在平均 Dice 上 OncoSeg 与 UNet3D 基本**打成平手**（OncoSeg 在 49/96 名受试者上领先，UNet3D 在 47/96 上领先）。正确的说法是*“在参数量约少 5 倍的情况下与 UNet3D 相当”*，而非*“击败它”*。UNet3D 还在约 30 轮时被 OOM 杀掉了，所以两者的训练预算并不对等。

In [ ]:
# 在内核中运行，以便图像内联渲染。辅助函数：受保护的 PNG 显示，使用
# /content/oncoseg（Colab）路径，并为本地克隆提供 cwd 回退。
import os
from IPython.display import Image, display

REPO = "/content/oncoseg" if os.path.isdir("/content/oncoseg") else os.getcwd()

def show_fig(rel_path, caption=None, width=1200):
    full = os.path.join(REPO, rel_path)
    if os.path.isfile(full):
        if caption:
            print(caption)
        display(Image(filename=full, width=width))
    else:
        print(f"[未找到图像：{full}]")

print("### 3.1 - 真实验证脑部 MRI 上的分割：OncoSeg 对比 UNet3D\n")
print("行 = 3 个代表性病例（最差 Dice=0.239 / 中位数=0.852 / 最佳=0.946）。")
print("列 = FLAIR 输入 | 专家真值标注 | OncoSeg | UNet3D 基线（19.2M 参数）。")
print("RGB 叠加：红色=ET（强化区）| 绿色=WT（整个肿瘤）| 蓝色=TC（肿瘤核心）。\n")
show_fig("figures/qualitative_comparison.png", None, width=1400)
print("\n真实 MRI 切片，非合成。最差病例显示模型在严重坏死的肿瘤上")
print("可能失败；中位数/最佳病例显示它已学到稳健的 3D 特征。")

### 8.2 · 它的准确度如何？

OncoSeg 在 BRATS 上（**n=96** 个验证受试者）达到了具有竞争力的 Dice，同时参数量比 UNet3D **少约 5.2 倍**（3.7M vs 19.2M），并且边界误差**更低**（HD95 15.35 mm vs 21.03 mm）。

- **各区域 Dice（OncoSeg）：** TC 0.790 · WT 0.853 · ET 0.748 · 均值 0.797
- **平均 Dice：** 0.797（OncoSeg）vs 0.794（UNet3D）—— 一个**统计学上的平局**
- **HD95 均值：** 15.35 mm（OncoSeg）vs 21.03 mm（UNet3D）
- **参数量：** 3.7M vs 19.2M

**诚实说明（F01）。** Wilcoxon 符号秩检验（单侧，OncoSeg > UNet3D）在每个区域都不显著 —— **TC p=0.46，WT p=0.995，ET p=0.57，均值 p=0.41**。在平均 Dice 上，OncoSeg 在 **49/96** 个受试者上领先，UNet3D 在 **47/96** 个上领先（相当于抛硬币）。单次运行，没有多个随机种子或置信区间，且 UNet3D 在约 30 个 epoch 时因 OOM 被强制终止（预算不对等）。**结论：OncoSeg 以约 5 倍更小的规模与 UNet3D 打平** —— 这是一个有利的效率权衡，而非已证明的准确度胜出。在做出任何“更好”的论断之前，需要一个多种子、预算对等的对比。

In [ ]:
# §3.2 准确率：训练曲线 + 各区域 Dice 柱状图 + 指标表
# 直接由已提交的评估 JSON 构建。复用 §3.1 中的 REPO/show_fig。
import json
import pandas as pd

res = os.path.join(REPO, "experiments", "local_results")

print("### 3.2 - 准确率\n")
print("训练曲线（50 个 epoch）与各区域 Dice 对比：\n")
show_fig("experiments/local_results/training_curves.png", None, width=1000)
show_fig("experiments/local_results/dice_comparison.png", None, width=1000)

try:
    with open(os.path.join(res, "oncoseg_eval.json")) as f:
        onc = json.load(f)
    with open(os.path.join(res, "unet3d_eval.json")) as f:
        unet = json.load(f)

    def row(label, ok, uk, fmt):
        ov, uv = onc[ok], unet[uk]
        return [label, f"{ov:{fmt}}", f"{uv:{fmt}}", f"{ov - uv:+{fmt[1:]}}"]

    tbl = [
        row("Dice TC",   "eval_dice_tc",   "eval_dice_tc",   ".4f"),
        row("Dice WT",   "eval_dice_wt",   "eval_dice_wt",   ".4f"),
        row("Dice ET",   "eval_dice_et",   "eval_dice_et",   ".4f"),
        row("Dice mean", "eval_dice_mean", "eval_dice_mean", ".4f"),
        row("HD95 mean (mm)", "eval_hd95_mean", "eval_hd95_mean", ".2f"),
    ]
    tbl.append(["Parameters", "3.7M", "19.2M", "~5.2x smaller"])
    tbl.append(["Val subjects", str(onc["num_val_subjects"]), str(unet["num_val_subjects"]), ""])
    df = pd.DataFrame(tbl, columns=["Metric", "OncoSeg", "UNet3D", "delta (Onco-UNet)"])
    print("\n定量对比（来自已提交的评估 JSON）：\n")
    print(df.to_string(index=False))

    print("\n统计显著性（Wilcoxon 符号秩检验，单侧 OncoSeg>UNet3D）：")
    print("  TC p=0.46 | WT p=0.995 | ET p=0.57 | mean p=0.41  -> 均不显著（F01）")
    print("  平均 Dice：OncoSeg 在 49/96 上胜出，UNet3D 在 47/96 上胜出 -> 打平，而非取胜。")
    print("  单次 50-epoch 运行；UNet3D 在约 30 个 epoch 时因 OOM 被终止 -> 预算不对等。")
except FileNotFoundError as e:
    print(f"[未找到评估 JSON: {e}] - 已跳过指标表。")

### 8.3 · 不确定性与可信度（MC-Dropout）

OncoSeg 使用 **Monte-Carlo Dropout** 估计每个体素的不确定性（5 次随机前向传播；逐通道的**二值**熵，因此其上界为 ln 2 ≈ 0.69 nats——发现 F16）。下面给出三个视图：

1. **不确定性图**（中位数病例 BRATS_425）：FLAIR、真实掩膜、MC-Dropout 熵热力图，以及预测误差叠加图。不确定性**集中在肿瘤边界**——即模型真正不确定的区域。
2. **可靠性图**（15 分箱 ECE）：预测置信度与经验准确率的对比。
3. **不确定性 vs 误差**：如预期，较高的熵对应较高的逐体素误差。

**校准告诫（F02）。** **合并后的 ECE 为 0.0101** 看似出色，但这是一个背景伪影——约 790 万个背景体素落在第一个分箱中（置信度 ~0，准确率 ~0.002），主导了整体平均值。若仅限于肿瘤体素，则**前景 ECE 为 0.49**：模型在**其误判的病灶体素上明显过于自信**。临床上：使用不确定性图来标记边界区域以供人工复核；**不要**将肿瘤体素的高置信度解读为正确性的保证。与 §3.1–§3.2 一样，这些均为来自单次运行的静态已提交图像。

In [ ]:
# §3.3 不确定性：三张已提交的图。复用 §3.1 中的 REPO/show_fig。
print("### 3.3 - 不确定性量化（MC-Dropout，5 个采样）\n")

figs = [
    ("figures/uncertainty_map.png",
     "不确定性图（BRATS_425）：熵集中在肿瘤边界处。"),
    ("figures/uncertainty_calibration.png",
     "可靠性图：整体 ECE=0.0101（受背景主导），但"
     "前景 ECE=0.49 -> 在肿瘤体素上过于自信（F02）。"),
    ("figures/uncertainty_vs_error.png",
     "不确定性 vs 误差：更高的 MC 熵对应更高的单体素误差。"),
]
for rel, cap in figs:
    print("\n" + cap)
    show_fig(rel, None, width=1000)

print("\n要点：不确定性有助于标记边界，但在肿瘤体素上的校准")
print("很差（前景 ECE=0.49）——而这些恰恰是最重要的体素。")

## 9 · 完整测试套件

以子进程方式运行（`!pytest`），因此使用刚刚安装的软件包。由于已安装 `dev,serve,dicom` 附加依赖，那些在裸机环境下会*跳过*的测试（monai/nibabel/pydicom/highdicom）现在会**真正运行**。预期结果为**约 194 项通过，0 项失败**。


In [ ]:
!cd /content/oncoseg && pytest tests/ -q -rs --tb=short

## 10 · Lint (ruff) — CI 运行的同一项检查


In [ ]:
!cd /content/oncoseg && ruff check src/ tests/ && echo 'ruff: 检查通过'

## 11 · 对已修复的算法代码路径进行冒烟测试（真实张量，无需数据集）

以**子进程**方式运行（全新的解释器）。在随机权重 + 合成体数据上，验证每条已修复的路径是否*能够运行*：

- **F04** 在已训练的内联 `train_all.OncoSeg` 上运行 MC-Dropout。
- **F16** 不确定性是逐通道的**二值**熵，上界为 `ln 2 ≈ 0.693`。
- **F06** RECIST 最长径扫描**所有**切片。
- **F09** `DeepSupervisionLoss` 对多尺度预测进行插值。
- **F08** 最优检查点的选择是 **NaN 安全的**。


In [ ]:
smoke = r'''import sys, os
# train_all.py 是位于仓库根目录的脚本（不是已安装的包模块），因此在导入它之前
# 先把仓库路径加入 sys.path。
sys.path.insert(0, os.getcwd())
import torch, numpy as np, math
device = "cuda" if torch.cuda.is_available() else "cpu"
print("设备:", device, "| numpy", np.__version__, "| torch", torch.__version__)
ok = True

# F04 + F16：在内联 train_all 架构（即已训练的那个）上运行 MC-Dropout
from train_all import OncoSeg as InlineOncoSeg
from src.inference import Predictor
model = InlineOncoSeg(in_channels=4, num_classes=3, embed_dim=24, depths=(2,2,2,2),
                      num_heads=(3,6,12,24), deep_supervision=False).to(device).eval()
assert hasattr(model, "decoders") and not hasattr(model, "decoder")
pred = Predictor(model=model, device=torch.device(device), roi_size=(64,64,64), mc_samples=4)
unc = pred._estimate_uncertainty(torch.rand(1,4,64,64,64, device=device))  # F04：不得抛出异常
f16 = unc.max() <= math.log(2) + 1e-3
print(f"F04 MC-dropout 已运行（形状 {unc.shape}）-> OK")
print(f"F16 熵<=ln2？ max={float(unc.max()):.4f} (ln2={math.log(2):.4f}) -> {'OK' if f16 else 'FAIL'}"); ok &= f16

# F06：跨所有切片的 RECIST 最长径
from src.response.recist import RECISTMeasurer
m = RECISTMeasurer()
mask = np.zeros((64,64,8), np.uint8); mask[10:40,10:40,0]=1; mask[30,5:55,1]=1
d = m.longest_axial_diameter(mask, pixdim=(1.0,1.0,1.0))
f06 = d > 45
print(f"F06 最长径={d:.1f}mm（预期约 49，修复前约 41）-> {'OK' if f06 else 'FAIL'}"); ok &= f06

# F09：深监督损失对多尺度预测进行插值
from src.training.losses import DeepSupervisionLoss, DiceCELoss
ds = DeepSupervisionLoss(DiceCELoss())
tgt = torch.zeros(1,3,32,32,32, device=device); tgt[:,0]=1
preds = [torch.randn(1,3,32,32,32, device=device), torch.randn(1,3,16,16,16, device=device), torch.randn(1,3,8,8,8, device=device)]
loss = ds(preds, tgt)  # F09：不得抛出形状错误
f09 = bool(torch.isfinite(loss)) and loss.dim()==0
print(f"F09 深监督损失={float(loss):.4f} 为有限标量 -> {'OK' if f09 else 'FAIL'}"); ok &= f09

# F08：NaN 安全的最佳检查点选择
guarded = lambda metric, best: (not math.isnan(metric)) and metric > best
row = np.array([0.71, 0.67, np.nan])  # 空 ET 的受试者 -> NaN 区域
f08 = math.isnan(float(np.mean(row))) and guarded(float(np.nanmean(row)), 0.0)
print(f"F08 普通均值为 NaN，nanmean={float(np.nanmean(row)):.4f}，带保护则保存 -> {'OK' if f08 else 'FAIL'}"); ok &= f08

print("\nSMOKE_RESULT:", "全部通过" if ok else "部分失败")
sys.exit(0 if ok else 1)
'''
with open('/content/_smoke.py','w') as f: f.write(smoke)
!cd /content/oncoseg && python /content/_smoke.py

## 12 · 从提交的数组中重新推导 CRITICAL 统计量

无需模型——直接从提交的 `.npy` / JSON 中重新计算文档报告的诚实数字（F01 Wilcoxon、F02 前景 ECE、F07 主导失败区域）。以子进程运行。


In [ ]:
stats = r'''import numpy as np, json
from scipy.stats import wilcoxon
o = np.load("experiments/local_results/oncoseg_per_subject_dice.npy")  # 列 [TC, WT, ET]
u = np.load("experiments/local_results/unet3d_per_subject_dice.npy")
print("每个受试者的数组:", o.shape, "(验证集 n =", o.shape[0], "-> 划分 388/96)")
for i,name in enumerate(["TC","WT","ET"]):
    a,b = o[:,i], u[:,i]; mk = ~(np.isnan(a)|np.isnan(b)); a,b = a[mk], b[mk]
    p = wilcoxon(a, b, alternative="greater").pvalue
    print(f"  {name}: delta={(a-b).mean():+.4f}  p(OncoSeg>UNet3D)={p:.4f}  OncoSeg 胜出 {int((a>b).sum())}/{int(mk.sum())}")
om, um = np.nanmean(o,axis=1), np.nanmean(u,axis=1); mm = ~(np.isnan(om)|np.isnan(um))
print("  平均 p =", round(float(wilcoxon(om[mm],um[mm],alternative="greater").pvalue),4), "-> F01: 没有区域显著; WT 偏向 UNet3D")
means = np.nanmean(o, axis=1); bottom = np.argsort(means)[:5]
opr, bpr = np.nanmean(o,axis=0), np.nanmean(o[bottom],axis=0)
rel = {n:(opr[i]-bpr[i])/opr[i] for i,n in enumerate(["TC","WT","ET"])}
print("  F07 相对下降 (最差 5 例):", {k:round(v,3) for k,v in rel.items()}, "-> 主导 =", max(rel, key=rel.get))
d = json.load(open("experiments/local_results/uncertainty_metrics.json"))
print("  F02 合并 ECE =", d["ece_median_case"], "| 前景 ECE =", d.get("ece_median_case_foreground"), "-> 对肿瘤过度自信")
'''
with open('/content/_stats.py','w') as f: f.write(stats)
!cd /content/oncoseg && python /content/_stats.py

## 13 · 见证它的运行 —— 自动化肿瘤追踪 → RECIST 1.1 疗效评估（附图）

内联渲染（在内核中运行）。将一个**基线**掩膜 + 三个**随访**掩膜（缩小 / 稳定 / 增大）送入 OncoSeg 的**真实** `RECISTMeasurer` + `ResponseClassifier`，并以结果表格和**前后对比图**的形式展示评估结论（CR/PR/SD/PD）（上图：基线为蓝色，随访为橙色，白色 = 重叠区域；下图：随访分割结果 + 评估结论）。

> **诚实说明。** 这些掩膜是**合成的球形体模**，其尺寸经过设计以跨越 RECIST 阈值，因此评估结论是*按构造*正确的 —— 它检验的是**测量 → 分类**的代码路径，而非模型精度。完全相同的 `classify()` 也在真实的 OncoSeg 分割结果上运行（`notebooks/recist_response_demo.ipynb`）。分割网络在 §11 中被检验；一次真正的端到端预测需要你先训练一个检查点（§14）。


In [ ]:
# 在内核中运行（而非子进程），这样图像会内联显示在下方。
import os, sys
sys.path.insert(0, "/content/oncoseg" if os.path.isdir("/content/oncoseg") else os.getcwd())
import numpy as np
import scipy.ndimage as ndi
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from src.response.recist import RECISTMeasurer
from src.response.classifier import ResponseClassifier, ResponseCategory

# 已知半径的合成球形“肿瘤”掩膜——几何仿真体，并非
# 模型预测结果。目的是展示真实的 RECIST 1.1 测量 +
# CR/PR/SD/PD 分类器在掩膜上端到端地运行。
def sphere(dim=80, r=14):
    m = np.zeros((dim, dim, dim), np.uint8)
    c = dim // 2
    zz, yy, xx = np.ogrid[:dim, :dim, :dim]
    m[(zz - c) ** 2 + (yy - c) ** 2 + (xx - c) ** 2 <= r ** 2] = 1
    return m

clf, meas, pix = ResponseClassifier(), RECISTMeasurer(), (1.0, 1.0, 1.0)
baseline = sphere(r=14)                                  # 约 28 mm 的靶病灶
scenarios = {"Shrinking": sphere(r=9), "Stable": sphere(r=13), "Growing": sphere(r=19)}
arrow = {"Shrinking": "↓", "Stable": "≈", "Growing": "↑"}
glyph = {"Partial Response": "PR", "Stable Disease": "SD",
         "Progressive Disease": "PD", "Complete Response": "CR"}
colr  = {"Partial Response": "#1f9e89", "Stable Disease": "#b8860b",
         "Progressive Disease": "#d1362f", "Complete Response": "#1f9e89"}

base_les = meas.measure_lesions(baseline, pix)
base_sld = sum(l["longest_diameter_mm"] for l in base_les)
print("=== OncoSeg RECIST 1.1 疗效反应演示（合成仿真体，真实测量 + 分类器）===")
print(f"基线：{len(base_les)} 个靶病灶，最长径之和 = {base_sld:.1f} mm\n")
hdr = f"{'scenario':11s}{'baseline':>10s}{'follow-up':>11s}{'change':>9s}   verdict"
print(hdr); print("-" * len(hdr))
rows = []
for name, fu in scenarios.items():
    r = clf.classify(baseline, fu, pixdim=pix)
    rows.append((name, fu, r))
    v = r.category.value
    print(f"{name:11s}{r.baseline_sum_ld:8.1f}mm{r.followup_sum_ld:9.1f}mm{r.percent_change*100:+8.1f}%   {glyph[v]}  {v}")

def edge(slc):
    return ndi.binary_dilation(slc, iterations=1) & ~slc.astype(bool)

mid = baseline.shape[2] // 2
bs = baseline[:, :, mid]
fig, axes = plt.subplots(2, 4, figsize=(16, 9.5))
# 第 0 列：基线参考 + 图例
axes[0, 0].imshow(bs, cmap="gray"); axes[0, 0].contour(edge(bs), colors="#7CF6C8", linewidths=1.4)
axes[0, 0].set_title(f"BASELINE\nSLD = {base_sld:.0f} mm", fontsize=12, weight="bold"); axes[0, 0].axis("off")
axes[1, 0].axis("off")
axes[1, 0].legend(handles=[
    Patch(facecolor="#9ecae1", label="baseline tumor"),
    Patch(facecolor="#fdae6b", label="follow-up tumor"),
    Patch(facecolor="white", edgecolor="#999", label="overlap (unchanged)")],
    loc="center", fontsize=11, frameon=False, title="Top row = before/after overlay")
for j, (name, fu, r) in enumerate(rows, start=1):
    fs = fu[:, :, mid]
    ov = np.zeros((*bs.shape, 3))
    ov[bs > 0] = [0.62, 0.79, 0.88]                 # 基线 = 蓝色
    ov[fs > 0] = [0.99, 0.68, 0.42]                 # 随访 = 橙色
    ov[(bs > 0) & (fs > 0)] = [1, 1, 1]             # 重叠 = 白色
    axes[0, j].imshow(ov); axes[0, j].axis("off")
    axes[0, j].set_title(f"{name}  {arrow[name]}", fontsize=12, weight="bold")
    axes[1, j].imshow(fs, cmap="gray"); axes[1, j].contour(edge(fs), colors="#7CF6C8", linewidths=1.4)
    v = r.category.value
    axes[1, j].set_title(f"{glyph[v]}   {r.percent_change*100:+.0f}% SLD\n{v}",
                         fontsize=12.5, color=colr[v], weight="bold")
    axes[1, j].axis("off")
fig.suptitle("OncoSeg — automated 3D tumor tracking & RECIST 1.1 treatment-response",
             fontsize=15, weight="bold", y=1.0)
fig.text(0.5, 0.03,
         "Top: baseline (blue) vs follow-up (orange); white = unchanged overlap.   "
         "Bottom: follow-up segmentation + automated verdict.\n"
         "RECIST 1.1:  PR = shrink ≥30%   ·   PD = grow ≥20% (and ≥5 mm)   ·   SD = in between.   "
         "Synthetic phantoms — the identical code runs on real OncoSeg segmentations.",
         ha="center", fontsize=10, color="#555")
fig.subplots_adjust(hspace=0.28)
fig.tight_layout(rect=[0, 0.07, 1, 0.95])
plt.show()

exp = {"Shrinking": ResponseCategory.PR, "Stable": ResponseCategory.SD, "Growing": ResponseCategory.PD}
print("\nDEMO_RESULT:", "ALL VERDICTS CORRECT" if all(r.category == exp[n] for n, _, r in rows) else "MISMATCH")

## 14 · （可选，较慢）端到端训练若干个 epoch

取消注释即可在真实的 MSD Brain Tumour 数据集上运行**训练循环**。这会下载 **~7 GB** 数据，并在 GPU 上训练几个 epoch（耗时数十分钟）。它用于验证已设定随机种子、带 NaN 防护、标签正确的训练路径能够端到端跑通；它**不会**复现论文中 50 个 epoch 的数值。


In [ ]:
# # 警告：会下载约 7GB 数据并进行训练。取消注释以运行。
# !cd /content/oncoseg && python train_local.py --epochs 2 --val-interval 1 --seed 42

## 15 · 回顾、诚实的局限性与后续方向

# 结语：你刚刚看到了什么 + 局限性 + 后续方向

## 完整闭环：从患者扫描到治疗反应

你刚刚走完了**一条完整的流水线**，用于自动化肿瘤分割和临床反应评估：

1. **环境搭建与数据**（§1–2）：从 Medical Segmentation Decathlon 加载 4 通道脑部 MRI（388 例训练 / 96 例验证），包含数据增强以及 3 通道的 BraTS 标签约定（TC = 肿瘤核心，WT = 整个肿瘤，ET = 增强肿瘤），按 [TC, WT, ET] 堆叠。

2. **架构**（§3）：探索了 **OncoSeg 混合设计**：
   - **编码器**：3D Swin Transformer，embed_dim=24（训练默认值），4 个阶段，dims=[24,48,96,192]，每阶段 patch-merge 下采样 ×2，采用 7×7×7 窗口的窗口化自注意力
   - **解码器**：CNN 上采样块配合**交叉注意力跳跃连接（Cross-Attention Skip connections）**——解码器查询编码器特征，而非盲目拼接。应用于中间跳跃连接（解码过程中的第 1、2、3 阶段）。
   - **损失**：DiceCELoss = 0.5×Dice + 0.5×BCEWithLogits（sigmoid 多标签）处理类别不平衡并提供稳定的梯度
   - **深监督（Deep Supervision）**：训练时在中间解码器尺度上设置辅助头，权重 = (1/2^i) 归一化到总和为 1；更深的尺度（更粗的预测）获得更低的权重
   - **MC-Dropout**：测试时随机 dropout（N 次前向传播），用于估计每个体素的熵作为不确定性信号；每个通道的二值熵 H = -(p log p + (1-p) log(1-p))

3. **构建与训练**：调用了 `train_all.py` 中确切的 PyTorch 模块（50 个 epoch，AdamW，余弦退火，梯度裁剪，roi_size=96³）。OncoSeg **3.7M 参数**（embed_dim=24）。

4. **结果画廊**（§4）：展示了在真实测试案例上的分割掩膜、一张 **Dice 精度表**，以及集中在肿瘤边界上的不确定性图——正是放射科医生会标记出来审阅的地方。

5. **实时 RECIST 演示**（§5）：端到端的临床终点：提取每个病灶的最长轴向直径，计算最长直径之和（SLD），根据 RECIST 1.1 阈值分类反应（CR/PR/SD/PD）。在合成的随访场景上产生了正确的判定。

6. **验证**（§6）：跨损失函数、交叉注意力、Swin 编码器、RECIST 测量、反应分类和校准运行了 **185 个单元测试**，以确保开发过程中没有任何东西被破坏。

---

## 局限性：请仔细阅读

### 研究设计

| 局限性 | 影响 | 详情 |
|-----------|--------|---------| 
| **单次运行、单个数据集** | 无泛化保证 | 所有精度数字均来自在 MSD Task01_BrainTumour 上的一次 50-epoch 训练（96 例验证，Apple M1 硬件）。跨数据集泛化（BraTS 2023、外部胶质瘤）未经评估。 |
| **相对于 UNet3D 无统计显著性** | 诚实的定位是参数效率，而非精度 | 对每个受试者 Dice 的 Wilcoxon 符号秩检验：TC p=0.46，WT p=0.995，ET p=0.57，均值 p=0.41。UNet3D 在 67/96 例受试者的 WT 上占优。平均 Dice 差异（+0.0025）处于运行间噪声范围内。27% 的 HD95 差距未经显著性检验（仅为汇总值）。 |
| **未提供已训练检查点** | 可复现性需要重新训练 | 已训练的 OncoSeg（embed_dim=24）未在仓库中分发；代码仅提供架构。在 M1 MPS 上于 MSD 上重新训练约需 12 小时。 |
| **前景校准较差** | 不确定性图可用于分诊，但不能作为已校准的概率 | 汇总的期望校准误差（ECE）= 0.0101，主要由背景（约 98.4% 的体素）主导。**仅在前景（肿瘤）体素上，ECE ≈ 0.49**；最高置信度分箱的正确率仅约 40%。模型在**肿瘤体素上过度自信**。请将熵图作为*相对*信号使用（熵越高 → 误差越高），而非绝对概率。 |
| **RECIST 演示为合成数据** | 并非临床验证 | 所有"随访"扫描都是对单一基线（BRATS_407，seed 42）的形态学扰动，经过调整以跨越 RECIST 阈值——因此 CR/PR/SD/PD 的判定在构造上是循环的。这仅验证了测量→分类的代码接线正确，而非它能在真实的纵向数据上工作。 |

### 已知失败模式

- **增强肿瘤（ET）是主要的失败区域**：在最差的 5 个案例上相对 Dice 下降 −84.3%（相比之下 TC 为 −79.7%，WT 为 −33.7%）。ET 体积小、依赖对比度，并且在困难案例中经常缺失。
- **小的、碎片化的、低对比度的肿瘤**：BRATS_077（最差案例，Dice 0.239）的肿瘤体积处于第 17.7 百分位，肿瘤-脑组织对比度弱（信号弱 3 倍），且形态碎片化。这些是固有的困难，而非 bug。
- **无消融研究**：4 项计划中的消融（无交叉注意力、无深监督、无 MC dropout、小 embed_dim）的框架已存在，但由于 GPU 可用性，仅在本地执行了空跑。

---

## 后续方向

### 关于架构与理论
- **交互式 3D 讲解**：[docs/oncoseg_explained.html](../docs/oncoseg_explained.html) —— 可旋转的 3D 架构图、每个模块的物理含义（"为什么需要每个模块"），以及一个带有具体张量形状的手工推演交叉注意力示例。
- **主 README**：[README.md](../README.md) —— 项目摘要、数据集概览、模型对比表（供参考的参数量）。

### 关于可复现性与实现细节
- **完整结果文档**：[docs/Paper_Results_Draft.md](../docs/Paper_Results_Draft.md) —— 分割精度、定性分析、失败模式案例研究（BRATS_077 诊断）、RECIST 流水线、不确定性校准细节、局限性清单。
- **训练框架**：`train_all.py` —— OncoSeg 内联定义 + UNet3D 基线；确切的损失、损失权重、学习率、调度器、数据变换。使用 `--embed-dim 24` 调用以匹配已训练模型。
- **测试套件**：`tests/`（20 个测试文件中的 185 个测试）—— 覆盖前向传播、深监督、损失函数、交叉注意力、Swin 编码器、RECIST 测量、反应分类和校准。

### 关于代码审查与透明度
- **31 项发现的审查 + 修复**：[docs/oncoseg_explained.html](../docs/oncoseg_explained.html) → **"Fixes"标签页** —— 来自独立审查的全部 31 项代码/数据/文档/配置发现，附有"Was → Fix → Verified"（原状 → 修复 → 已验证）台账、提交哈希以及重新计算的真值。总结："脚手架依然稳固；科学如今诚实。"

### 关于临床集成
- **RECIST 测量器**：`src/response/recist.py` —— 每个病灶的最长轴向直径、体积、SLD 计算。
- **反应分类器**：`src/response/response.py`（或 `classifier.py`）—— 按 RECIST 1.1 进行 CR/PR/SD/PD 分类。
- **DICOM 服务器**：`deploy/orthanc/` —— 用于 DICOM I/O 的 FastAPI 封装（生产测试待定）。

---

## 诚实的结论

**OncoSeg 是一个参数高效的 Swin+CNN 混合架构，在肿瘤分割上以约 5× 更少的参数（3.7M vs 19.2M）匹配了标准的 3D U-Net（无统计学差异，p=0.41）。** 不确定性图作为*相对*分诊信号是有用的，但在肿瘤体素上过度自信（前景 ECE ≈ 0.49）。完整的分割 → RECIST 流水线端到端运行，并在合成数据上产生了正确的反应判定；真正的纵向验证需要配对扫描的临床数据集。

这是一项针对脑肿瘤的**单次运行、单个数据集的研究**。优点：诚实的不确定性披露、模块化架构、可复现的训练框架、全面的测试覆盖（185 个测试，31 项发现已修复）。不足：未提供已训练检查点、相对于基线无统计显著性、前景校准较差、RECIST 演示为合成数据、无消融研究。

如果你想看看审查过程中发现了什么，请**阅读 Fixes 标签页**——所有补救措施都附有提交哈希和重新计算的真值文档。

In [ ]:

# 显示结尾章节（Markdown 单元格；无需执行代码）
# 本单元格主要用于教学目的，作为总结性收尾。
# 如果需要显示图表：

from IPython.display import Image, display

# 可选：再次渲染不确定性校准图作为提醒
calibration_path = "figures/uncertainty_calibration.png"
try:
    display(Image(calibration_path))
    print("校准提醒：前景 ECE 约为 0.49（在肿瘤体素上过于自信）")
except FileNotFoundError:
    import os
    if os.path.exists("figures/uncertainty_calibration.png"):
        display(Image("figures/uncertainty_calibration.png"))
    else:
        print("（未找到校准图；详情请参见 docs/Paper_Results_Draft.md）")
